In [ ]:
import build123d
import fenics
import gmsh
import meshio

import numpy as np

In [ ]:
joint_configuration = {
    "load": 60000,
    "desired_safety_factor": 3.0,
    "bolt_yield_strength": 940,
    "plate_yield_strength": 250,
    "preload": 150000,
    "pitch": 1.5,
    "plate_thickness": 10,
    "bolt_elastic_modulus": 210,
    "plate_elastic_modulus": 210,

    "num_bolts": 4,
    }

In [ ]:
plate_width = 0.1  # [m]
plate_length = 0.2  # [m]

plate_thickness = 10  # [mm]
plate_elastic_modulus = 210  # [GPa]
plate_yield_strength = 250  # [MPa]

load = 60000  # [N]
traction = -load / (plate_thickness / 1000 * plate_length)

num_bolts = 4
bolt_diameter = 10  # [mm]
bolt_elastic_modulus = 210  # [GPa]
bolt_yield_strength = 940  # [MPa]

In [ ]:
plate_thickness_m=plate_thickness / 1000
num_holes=num_bolts
elastic_modulus=plate_elastic_modulus * 10**9
yield_strength=plate_yield_strength * 10**6
traction_values=[(0, traction, 0)]
hole_radius_m=bolt_diameter / 2 / 1000
plate_length_m=plate_length
plate_width_m=plate_width
edge_margin_m=plate_length / (2 * num_bolts)
hole_spacing_m=plate_length / num_bolts
hole_offset_from_bottom_m=0.020  # [m] vertical position of hole centers (Y from bottom edge)
plate_gap_mm=0.01  # [mm] gap between the two plates
poissons_ratio=0.3 # Poisson's ratio for steel

# Test code

In [ ]:
import fea_solver_dual
from parametric_cad_solver import create_two_plate_assembly_components

In [ ]:
plateA,plateB,boltCylinders = create_two_plate_assembly_components(
    plate_length_m=plate_length_m,
    plate_width_m=plate_width_m,
    plate_thickness_m=plate_thickness_m,
    num_holes=num_holes,
    hole_radius_m=hole_radius_m,
    edge_margin_m=edge_margin_m,
    hole_spacing_m=hole_spacing_m,
    hole_offset_from_bottom_m=hole_offset_from_bottom_m,
    plate_gap_mm=plate_gap_mm
)

In [ ]:
fea_solver_dual._calculate_fos_from_build123d_dual(
    plateA,
    plateB,
    boltCylinders,
    elastic_modulus,
    poissons_ratio,
    bolt_yield_strength * 10**6,
    plate_yield_strength * 10**6,
    traction_values,
)

# Test old code

In [ ]:
import fea_solver
from parametric_cad_solver import create_two_plate_assembly

In [ ]:
object = create_two_plate_assembly(
    plate_length_m=plate_length_m,
    plate_width_m=plate_width_m,
    plate_thickness_m=plate_thickness_m,
    num_holes=num_holes,
    hole_radius_m=hole_radius_m,
    edge_margin_m=edge_margin_m,
    hole_spacing_m=hole_spacing_m,
    hole_offset_from_bottom_m=hole_offset_from_bottom_m,
    plate_gap_mm=plate_gap_mm
)

In [ ]:
fea_solver._calculate_fos_from_build123d(
    object,
    elastic_modulus,
    poissons_ratio,
    yield_strength,
    traction_values
)

# Create assembly step file

In [ ]:
# convert the gap to meters
gap_m = plate_gap_mm * 1e-4

# 1) Build Plate A (with through‑holes)
with build123d.BuildPart() as bp:
    build123d.Box(plate_length_m, plate_width_m, plate_thickness_m)
    # center the box around the XY origin
    bp.part = bp.part.move(build123d.Pos(plate_length_m / 2, plate_width_m / 2, 0))

    # compute X‑coordinates of hole centers
    x0 = plate_length_m - edge_margin_m
    x_coords = [x0 - i * hole_spacing_m for i in range(num_holes)]

    # subtract each hole
    for x in x_coords:
        with build123d.Locations(build123d.Pos(x, hole_offset_from_bottom_m, 0)):
            build123d.Cylinder(
                radius=hole_radius_m,
                height=plate_thickness_m + 1e-3,  # +1 mm to ensure clean cut
                mode=build123d.Mode.SUBTRACT,
            )
plateA = bp.part

# 2) Copy Plate A → shift up by thickness+gap → mirror in Y → re‑position
plateB = build123d.copy(plateA)
# shift in Z by plate_thickness + gap
plateB = plateB.locate(
    build123d.Location((0, 0, plate_thickness_m + gap_m), (0, 0, 1), 0)
)
# mirror about the XZ plane (so y→−y)
plateB = build123d.mirror(plateB, about=build123d.Plane.XZ)
# then translate back up in Y by 2*hole_center_y to line holes up
plateB = plateB.locate(
    build123d.Location((0, 2 * hole_offset_from_bottom_m, 0), (0, 0, 1), 0)
)

# 3) Create the bolt‑like cylinders (conformal, centered in the holes)
bolt_height_m = 2 * plate_thickness_m + gap_m
z_offset_m = plate_thickness_m / 2 + gap_m / 2  # start mid‑thickness of Plate A

with build123d.BuildPart() as bp2:
    for x in x_coords:
        with build123d.Locations(
            build123d.Pos(x, hole_offset_from_bottom_m, z_offset_m)
        ):
            build123d.Cylinder(
                radius=hole_radius_m,
                height=bolt_height_m,
                align=(
                    build123d.Align.CENTER,
                    build123d.Align.CENTER,
                    build123d.Align.CENTER,
                ),
            )
bolt_cylinders = bp2.part

# 4) Assemble and return
assembly = build123d.Compound(children=[plateA, plateB, bolt_cylinders])

In [ ]:
assembly

# convert build123d mesh to gsmh mesh to fenics mesh

In [ ]:
import tempfile

In [ ]:
gmsh.initialize()

In [ ]:
gmsh.clear()

In [ ]:
tempdir = "temp"

step_file = tempdir + "/model.step"
build123d.export_step(assembly, step_file)

msh_file = tempdir + "/mesh_merged.msh"
xdmf_file_mesh = tempdir + "/mesh.xdmf"
xdmf_file_surface_tags = tempdir + "surface_tags.xdmf"

In [ ]:
def import_step_and_capture_new_volumes(step_path: str) -> list[int]:
    """Import a STEP file and return the list of volume entity tags created by that import."""
    before = set(t for _, t in gmsh.model.getEntities(3))
    gmsh.merge(step_path)
    gmsh.model.occ.synchronize()
    after = set(t for _, t in gmsh.model.getEntities(3))
    new_tags = sorted(after - before)
    return new_tags

In [ ]:
tempdir = tempfile.TemporaryDirectory()

# --- Export STEP files (no geometry parameter changes) ---
step_plateA = tempdir.name + "/plateA.step"
step_plateB = tempdir.name + "/plateB.step"
step_bolts = tempdir.name + "/bolts.step"
build123d.export_step(plateA, step_plateA)
build123d.export_step(plateB, step_plateB)
build123d.export_step(bolt_cylinders, step_bolts)

In [ ]:
plateA_vols = import_step_and_capture_new_volumes(step_plateA)
plateB_vols = import_step_and_capture_new_volumes(step_plateB)
bolt_vols = import_step_and_capture_new_volumes(step_bolts)

In [ ]:
in_dimtags = [(3, t) for t in (plateA_vols + plateB_vols + bolt_vols)]
_, out_map = gmsh.model.occ.fragment(in_dimtags, [])
gmsh.model.occ.synchronize()

gmsh.model.occ.removeAllDuplicates()
gmsh.model.occ.synchronize()

In [ ]:
bolt_set_pre = set(bolt_vols)

bolt_frag_vols: set[int] = set()
plate_frag_vols: set[int] = set()

In [ ]:
for (src_dim, src_tag), frags in zip(in_dimtags, out_map):
    frag_vols = [t for dim, t in frags if dim == 3]
    if src_tag in bolt_set_pre:
        bolt_frag_vols.update(frag_vols)
    else:
        plate_frag_vols.update(frag_vols)

In [ ]:
BOLT_TAG = 101
PLATE_TAG = 102
gmsh.model.addPhysicalGroup(3, sorted(bolt_frag_vols), tag=BOLT_TAG)
gmsh.model.setPhysicalName(3, BOLT_TAG, "bolt")
gmsh.model.addPhysicalGroup(3, sorted(plate_frag_vols), tag=PLATE_TAG)
gmsh.model.setPhysicalName(3, PLATE_TAG, "plate")

In [ ]:
surfaces = gmsh.model.getEntities(2)

In [ ]:
for (sdim, stag) in surfaces:
    gmsh.model.addPhysicalGroup(sdim, [stag], tag=stag)

In [ ]:
surface_info = []
for dim, s in gmsh.model.getEntities(2):
    xmin, ymin, zmin, xmax, ymax, zmax = gmsh.model.occ.getBoundingBox(dim, s)
    surface_info.append((s, ymin, ymax))

In [ ]:
global_ymin = min(ymin for _, ymin, _ in surface_info)
global_ymax = max(ymax for _, _, ymax in surface_info)

In [ ]:
def y_face(target_y: float, tol: float = 1e-6) -> int:
    for tag, ymin, ymax in surface_info:
        if abs(ymax - ymin) < tol and abs(ymin - target_y) < tol:
            return tag
    raise RuntimeError(f"Could not find planar Y-face at y={target_y}")

In [ ]:
trac_pos_tag = y_face(global_ymin)
trac_neg_tag = y_face(global_ymax)

In [ ]:
zero_disp_tags = [trac_pos_tag]
traction_tags = [trac_neg_tag]

In [ ]:
gmsh.option.setNumber("Mesh.Algorithm3D", 10)  # or 4 if 10 not supported
gmsh.option.setNumber("Mesh.Optimize", 1)
gmsh.option.setNumber("Mesh.OptimizeNetgen", 1)

gmsh.model.mesh.generate(3)

gmsh.model.mesh.optimize("Netgen")

gmsh.model.mesh.removeDuplicateNodes()
gmsh.write(msh_file)

In [ ]:
gmsh.finalize()

In [ ]:
msh = meshio.read(msh_file)

In [ ]:
def to_xdmf(mesh, cell_type: str, out_path: str):
    cells = mesh.get_cells_type(cell_type)
    cell_data = mesh.get_cell_data("gmsh:physical", cell_type)
    out = meshio.Mesh(
        points=mesh.points,
        cells={cell_type: cells},
        cell_data={"name_to_read": [cell_data]},
    )
    meshio.write(out_path, out)

In [ ]:
to_xdmf(msh, "tetra", xdmf_file_mesh)
to_xdmf(msh, "triangle", xdmf_file_surface_tags)

In [ ]:
mesh = fenics.cpp.mesh.Mesh()
with fenics.cpp.io.XDMFFile(xdmf_file_mesh) as infile:
    infile.read(mesh)

In [ ]:
mvc_surf = fenics.MeshValueCollection("size_t", mesh, 2)
with fenics.cpp.io.XDMFFile(xdmf_file_surface_tags) as infile:
    infile.read(mvc_surf, "name_to_read")
mf = fenics.cpp.mesh.MeshFunctionSizet(mesh, mvc_surf)

In [ ]:
mvc_cell = fenics.MeshValueCollection("size_t", mesh, 3)
with fenics.XDMFFile(xdmf_file_mesh) as infile:
    infile.read(mvc_cell, "name_to_read")
cf = fenics.MeshFunction("size_t", mesh, mvc_cell)

# FOS calculation

In [ ]:
V = fenics.VectorFunctionSpace(mesh, "CG", 2)

bcs = [
    fenics.DirichletBC(V, fenics.Constant((0.0, 0.0, 0.0)), mf, tag)
    for tag in zero_disp_tags
]

mu = elastic_modulus / (2.0 * (1.0 + poissons_ratio))
lmbda = elastic_modulus * poissons_ratio / (
    (1.0 + poissons_ratio) * (1.0 - 2.0 * poissons_ratio)
)

In [ ]:
def epsilon(u):
    return 0.5 * (fenics.grad(u) + fenics.grad(u).T)

def sigma(u):
    return 2.0 * mu * epsilon(u) + lmbda * fenics.tr(epsilon(u)) * fenics.Identity(3)

In [ ]:
u = fenics.TrialFunction(V)
v = fenics.TestFunction(V)
f = fenics.Constant((0.0, 0.0, 0.0))

In [ ]:
a = fenics.inner(sigma(u), epsilon(v)) * fenics.dx

In [ ]:
ds = fenics.Measure("ds", domain=mesh, subdomain_data=mf)

In [ ]:
L = fenics.dot(f, v) * fenics.dx

In [ ]:
for tag, traction in zip(traction_tags, traction_values):
    L += fenics.dot(fenics.Constant(traction), v) * ds(tag)

In [ ]:
u_sol = fenics.Function(V)
fenics.solve(a == L, u_sol, bcs)

In [ ]:
s_dev = sigma(u_sol) - (1.0 / 3.0) * fenics.tr(sigma(u_sol)) * fenics.Identity(3)

In [ ]:
W0 = fenics.FunctionSpace(mesh, "DG", 0)

In [ ]:
vm0 = fenics.project(fenics.sqrt(3.0 / 2.0 * fenics.inner(s_dev, s_dev)), W0)

In [ ]:
vals = vm0.vector().get_local()
markers = cf.array()

In [ ]:
bolt_vals = vals[markers == BOLT_TAG]
plate_vals = vals[markers == PLATE_TAG]

In [ ]:
import numpy as np

In [ ]:
vm0.vector().max()

In [ ]:
max_bolt_vm = float(np.max(bolt_vals))
max_plate_vm = float(np.max(plate_vals))

In [ ]:
max_plate_vm

In [ ]:
max_bolt_vm

In [ ]:
bolt_fos = float(bolt_yield_strength / max_bolt_vm)
plate_fos = float(plate_yield_strength / max_plate_vm)

In [ ]:
print(bolt_fos, plate_fos)